[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/verification/schafer-turek-2d2.ipynb)

# **Schäfer-Turek 2D-2 benchmark**
**Johan Hoffman**

# **Abstract**

The GLS stabilized P1/P1 Navier-Stokes solver is applied to the Schäfer-Turek 2D-2 benchmark (unsteady Re=100 flow past a cylinder in a channel, Schäfer & Turek 1996) using a graded mesh (gmsh_rect_minus_circles v6, dist_min=0.5D=0.05 m, dist_max=3D=0.30 m). dt = time_step(msh, U_ref=U_m=1.5, C_CFL=0.5) = 0.5·h_min/1.5. Three refinement levels (res=32/64/128) are run to T=15 s; the last five complete lift periods give C_D,max, C_L,max, St = f·D/U_mean, and Δp (at t₀ + T_lift/2). ST96 reference intervals: C_D,max∈[3.22,3.24], C_L,max∈[0.99,1.01], St∈[0.295,0.305], Δp∈[2.46,2.50].

L1 (res=32, 2751 cells, dt=0.00238, 6301 steps, 230 s): C_D,max=3.290 ✗, C_L,max=0.778 ✗, St=0.2918 ✗, Δp=2.522 ✗. Coarse mesh severely underestimates C_L,max amplitude.

L2 (res=64, 10716 cells, dt=0.00117, 12850 steps, 1938 s): C_D,max=3.268 ✗, C_L,max=1.035 ✗, St=0.2986 ✓, Δp=2.505 ✗. St enters interval at L2; C_D,max, C_L,max, and Δp overshoot their ST96 upper bounds.

L3 (res=128, 42601 cells, dt=0.000520, 28847 steps, 114797 s wall-clock): C_D,max=3.241 ✗, C_L,max=1.001 ✓, St=0.3011 ✓, Δp=2.488 ✓. Three of four QoIs enter the ST96 interval at L3. C_D,max decreases monotonically and reaches 3.241, just above the upper bound 3.240.

# **About the code**

In [ ]:
# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)
# SPDX-License-Identifier: MIT
#
# Verification notebook: Schäfer-Turek 2D-2 unsteady benchmark.
# This template is maintained by Johan Hoffman.
# Please report problems to jhoffman@kth.se

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
import time
import ufl
import basix.ufl as bufl
from mpi4py import MPI
from matplotlib import pyplot as plt

import dolfinx
from dolfinx import fem
from dolfinx.fem import functionspace, Function, form, assemble_scalar, Constant
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]x[0,H] minus regular-polygon holes."""
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]x[0,H] rectangle with holes.

    left=1, right=2, lower=3, upper=4, objects=5.
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_scalar(f, title="Scalar field", ax=None):
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)
    ldm = V1v.dofmap.list
    arr = u1v.x.array.real
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: time_step v1 ---
import numpy as np


def time_step(msh, U_ref, C_CFL=0.5):
    """Return C_CFL * h_min / U_ref (CFL-based time step)."""
    tdim = msh.topology.dim
    n_cells = msh.topology.index_map(tdim).size_local
    h_min = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
    return C_CFL * h_min / U_ref
# --- end canonical: time_step ---

# **Introduction**

The Schäfer-Turek 2D-2 benchmark describes unsteady laminar flow past a circular cylinder in a channel at Re = 100 (Schäfer & Turek 1996). The domain is [0, 2.2] × [0, 0.41] with a cylinder of radius r = 0.05 centred at (0.2, 0.2). A parabolic inflow profile with maximum velocity U_m = 1.5 (mean U = 1.0) is prescribed at the left boundary; no-slip is applied on the upper and lower channel walls and on the cylinder; pressure p = 0 is set at the outflow. The Reynolds number is Re = U_mean · D / ν = 1.0 · 0.1 / 1e-3 = 100.

The flow is unsteady and develops periodic vortex shedding. The simulation is run to T = 15 s; the last five complete lift periods are used to extract the benchmark quantities. Reference intervals (ST96 2D-2): C_D,max ∈ [3.22, 3.24], C_L,max ∈ [0.99, 1.01], St ∈ [0.295, 0.305], Δp ∈ [2.46, 2.50] (Δp sampled at t₀ + T_lift/2 where t₀ is the time of maximum C_L in the analysis window).

The time step is dt = time_step(msh, U_ref = U_m, C_CFL = 0.5) = 0.5 · h_min / U_m, where h_min is the global minimum cell diameter. This uses the canonical CFL rule (time_step v1) applied with the maximum inflow velocity as the reference scale, consistently with the 2D-1 rerun.

# **Method**

**Parameters**

In [ ]:
# Problem geometry
L, H = 2.2, 0.41
cx, cy, r = 0.2, 0.2, 0.05
D = 2.0 * r           # cylinder diameter = 0.1 m

# Physical parameters
nu = 1e-3             # kinematic viscosity; Re = U_mean*D/nu = 1.0*0.1/0.001 = 100
U_m = 1.5             # max inflow velocity; mean U = 2/3*U_m = 1.0

# Simulation time
T = 15.0              # integration time
plot_freq = 500       # print progress every plot_freq steps

# Graded mesh parameters (v6)
dist_min_grade = 0.5 * D   # 0.05 m
dist_max_grade = 3.0 * D   # 0.30 m

# Reference values (Schäfer & Turek 1996, 2D-2)
C_D_max_ref = 3.2300
C_L_max_ref = 1.0000
St_ref      = 0.3000
dp_ref      = 2.4700

# Refinement levels: coarse, medium, fine
levels = [
    {"resolution": 32,  "segments": 32},
    {"resolution": 64,  "segments": 64},
    {"resolution": 128, "segments": 128},
]

**Convergence study**

In [ ]:
from dolfinx.geometry import (
    bb_tree as _bb_tree,
    compute_collisions_points as _ccp,
    compute_colliding_cells as _ccc,
)

U_mean    = (2.0 / 3.0) * U_m
force_ref = -2.0 / (U_mean**2 * D)   # C = force_ref * F (rho=1)

results = []

for lvl_idx, lvl in enumerate(levels):
    if lvl_idx > 0 and T < 1.0:   # fast mode: coarsest level only
        results.append(None)
        continue


    resolution = lvl["resolution"]
    segs       = lvl["segments"]
    print(f"\n--- Level {lvl_idx + 1}: resolution={resolution}, segments={segs} ---",
          flush=True)

    msh, _ = gmsh_rect_minus_circles(
        L, H, [(cx, cy, r)], resolution, segments=segs,
        dist_min=dist_min_grade, dist_max=dist_max_grade,
    )
    facet_tags = tag_boundaries(msh, L, H)

    tdim    = msh.topology.dim
    fdim    = tdim - 1
    n_cells = msh.topology.index_map(tdim).size_local

    dt_val = time_step(msh, U_ref=U_m, C_CFL=0.5)
    print(f"  cells={n_cells}, dt={dt_val:.6f}", flush=True)

    plot_mesh(msh, title=f"Level {lvl_idx + 1}: res={resolution}, cells={n_cells}")

    # Function spaces
    P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
    P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
    V   = functionspace(msh, P1v)
    Q   = functionspace(msh, P1s)
    V0, _ = V.sub(0).collapse()
    V1, _ = V.sub(1).collapse()

    inlet_facets  = facet_tags.find(1)
    cyl_facets    = facet_tags.find(5)
    wall_facets   = np.concatenate([facet_tags.find(3), facet_tags.find(4)])
    outlet_facets = facet_tags.find(2)

    def _zero_bc(sub_idx, sub_V, facets):
        f = Function(sub_V); f.x.array[:] = 0.0
        dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
        return fem.dirichletbc(f, dofs, V.sub(sub_idx))

    u_in_func = Function(V)
    u_in_func.interpolate(lambda x: np.vstack([
        4.0 * U_m * x[1] * (H - x[1]) / H**2,
        np.zeros_like(x[1]),
    ]))
    bcu_in   = fem.dirichletbc(u_in_func,
                               fem.locate_dofs_topological(V, fdim, inlet_facets))
    bcu_wall = [_zero_bc(0, V0, wall_facets), _zero_bc(1, V1, wall_facets)]
    bcu_cyl  = [_zero_bc(0, V0, cyl_facets),  _zero_bc(1, V1, cyl_facets)]
    bcu = [bcu_in] + bcu_wall + bcu_cyl
    bcp = [fem.dirichletbc(Constant(msh, PETSc.ScalarType(0.0)),
                           fem.locate_dofs_topological(Q, fdim, outlet_facets), Q)]

    u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
    p1 = Function(Q, name="p1")
    dt_c = Constant(msh, PETSc.ScalarType(dt_val))
    nu_c = Constant(msh, PETSc.ScalarType(nu))

    u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
    p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

    h_c   = ufl.CellDiameter(msh)
    u_mag = ufl.sqrt(ufl.dot(u1, u1))
    d1    = 1.0 / ufl.sqrt((1.0/dt_c)**2 + (u_mag/h_c)**2)
    d2    = h_c * u_mag
    um    = 0.5 * (u + u0)
    um1   = 0.5 * (u1 + u0)

    Fu = (ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1), v)*ufl.dx
        - p1*ufl.div(v)*ufl.dx
        + nu_c*ufl.inner(ufl.grad(um), ufl.grad(v))*ufl.dx
        + d1*ufl.inner((u - u0)/dt_c + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                       ufl.dot(ufl.grad(v), um1))*ufl.dx
        + d2*ufl.div(um)*ufl.div(v)*ufl.dx)

    Fp = (d1*ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                       ufl.grad(q))*ufl.dx
        + ufl.div(um1)*q*ufl.dx)

    a_u = form(ufl.lhs(Fu)); l_u = form(ufl.rhs(Fu))
    a_p = form(ufl.lhs(Fp)); l_p = form(ufl.rhs(Fp))

    # Volume-form force (Green's formula)
    def _make_psi(phi_x, phi_y):
        psi = Function(V); psi.x.array[:] = 0.0
        dx_dofs = fem.locate_dofs_topological((V.sub(0), V0), fdim, cyl_facets)
        dy_dofs = fem.locate_dofs_topological((V.sub(1), V1), fdim, cyl_facets)
        if len(dx_dofs): psi.x.array[dx_dofs[0]] = phi_x
        if len(dy_dofs): psi.x.array[dy_dofs[0]] = phi_y
        psi.x.scatter_forward()
        return psi

    def _make_force_form(psi_f):
        return form(
            ufl.inner((u1 - u0)/dt_c + ufl.dot(ufl.grad(um1), um1), psi_f)*ufl.dx
            - p1*ufl.div(psi_f)*ufl.dx
            + nu_c*ufl.inner(ufl.grad(um1), ufl.grad(psi_f))*ufl.dx
        )

    psi_drag  = _make_psi(1.0, 0.0)
    psi_lift  = _make_psi(0.0, 1.0)
    drag_form = _make_force_form(psi_drag)
    lift_form = _make_force_form(psi_lift)

    # Pressure probe at cylinder front/rear stagnation polygon vertices
    p_pts  = np.array([[cx - r, cy, 0.0],
                       [cx + r, cy, 0.0]])
    p_tree = _bb_tree(msh, msh.topology.dim)
    p_cand = _ccp(p_tree, p_pts)
    p_adj  = _ccc(msh, p_cand, p_pts)
    p_cidx = np.array(
        [p_adj.array[p_adj.offsets[i]] for i in range(len(p_pts))],
        dtype=np.int32,
    )

    A_u = create_matrix(a_u); b_u = create_vector(V)
    A_p = create_matrix(a_p); b_p = create_vector(Q)

    def _ksp(A, pc_type, amg=False):
        ksp = PETSc.KSP().create(msh.comm)
        ksp.setOperators(A); ksp.setType("bcgs")
        pc = ksp.getPC(); pc.setType(pc_type)
        if amg: pc.setHYPREType("boomeramg")
        ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
        return ksp

    ksp_u = _ksp(A_u, "ilu")
    ksp_p = _ksp(A_p, "hypre", amg=True)

    set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
    set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

    t_wall0 = time.time()
    t_arr = []; C_L_arr = []; C_D_arr = []; dp_arr = []
    t = dt_val; step = 0

    while t < T + 1e-10:
        for _ in range(5):
            A_u.zeroEntries()
            assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
            with b_u.localForm() as loc: loc.set(0.0)
            assemble_vector(b_u, l_u)
            apply_lifting(b_u, [a_u], bcs=[bcu])
            b_u.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_u, bcu)
            ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
            u1.x.scatter_forward()
            set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

            A_p.zeroEntries()
            assemble_matrix(A_p, a_p, bcs=bcp); A_p.assemble()
            with b_p.localForm() as loc: loc.set(0.0)
            assemble_vector(b_p, l_p)
            apply_lifting(b_p, [a_p], bcs=[bcp])
            b_p.ghostUpdate(addv=PETSc.InsertMode.ADD,
                            mode=PETSc.ScatterMode.REVERSE)
            set_bc(b_p, bcp)
            ksp_p.setOperators(A_p, A_p); ksp_p.solve(b_p, p1.x.petsc_vec)
            p1.x.scatter_forward()
            set_bc(p1.x.petsc_vec, bcp); p1.x.scatter_forward()

        F_drag = float(msh.comm.allreduce(assemble_scalar(drag_form), op=MPI.SUM))
        F_lift = float(msh.comm.allreduce(assemble_scalar(lift_form), op=MPI.SUM))
        C_D_t  = force_ref * F_drag
        C_L_t  = force_ref * F_lift
        p_vals = p1.eval(p_pts, p_cidx)
        dp_t   = float(p_vals[0, 0] - p_vals[1, 0])

        t_arr.append(t); C_D_arr.append(C_D_t)
        C_L_arr.append(C_L_t); dp_arr.append(dp_t)

        u0.x.array[:] = u1.x.array
        t += dt_val; step += 1

        if step % plot_freq == 0:
            print(f"  step={step}, t={t:.3f}, C_D={C_D_t:.4f}, C_L={C_L_t:.6f}",
                  flush=True)

    t_wall = time.time() - t_wall0
    print(f"  Done: {step} steps, wall={t_wall:.1f}s", flush=True)

    t_np   = np.array(t_arr)
    C_L_np = np.array(C_L_arr)
    C_D_np = np.array(C_D_arr)
    dp_np  = np.array(dp_arr)

    # Time-series plots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
    ax1.plot(t_np, C_L_np); ax1.set_xlabel("t"); ax1.set_ylabel("C_L")
    ax1.set_title(f"Lift coefficient — Level {lvl_idx + 1} (res={resolution})")
    ax1.axhline(0, color='k', linewidth=0.5)
    ax2.plot(t_np, C_D_np); ax2.set_xlabel("t"); ax2.set_ylabel("C_D")
    ax2.set_title(f"Drag coefficient — Level {lvl_idx + 1} (res={resolution})")
    plt.tight_layout(); plt.show()

    # Periodic analysis: upward zero crossings of C_L
    crossings = []
    for i in range(len(C_L_np) - 1):
        if C_L_np[i] <= 0 and C_L_np[i + 1] > 0:
            frac = -C_L_np[i] / (C_L_np[i + 1] - C_L_np[i])
            crossings.append(t_np[i] + frac * (t_np[i + 1] - t_np[i]))

    if len(crossings) < 6:
        print(f"  WARNING: {len(crossings)} upward crossings found; "
              f"T too short for periodic analysis", flush=True)
        results.append({"lvl": lvl_idx + 1, "res": resolution, "cells": n_cells,
                         "dt": dt_val, "steps": step, "wall": t_wall,
                         "ncross": len(crossings), "periodic": False})
        continue

    # Last 5 complete periods (6 crossings span 5 periods)
    t_start = crossings[-6]
    t_end   = crossings[-1]
    T_lift  = (t_end - t_start) / 5.0
    f_lift  = 1.0 / T_lift
    St_val  = f_lift * D / U_mean

    mask    = (t_np >= t_start) & (t_np <= t_end)
    C_D_max = float(np.max(C_D_np[mask]))
    C_L_max = float(np.max(C_L_np[mask]))

    # Δp at t0 + T_lift/2, where t0 is the time of maximum C_L in window
    t_masked = t_np[mask]
    i_cl_max = int(np.argmax(C_L_np[mask]))
    t0_cl    = float(t_masked[i_cl_max])
    t_dp     = t0_cl + 0.5 * T_lift
    dp_val   = float(np.interp(t_dp, t_np, dp_np))

    print(f"  T_lift={T_lift:.5f}, f={f_lift:.4f}, St={St_val:.4f}", flush=True)
    print(f"  C_D_max={C_D_max:.6f}, C_L_max={C_L_max:.6f}, dp={dp_val:.6f}",
          flush=True)

    # Velocity field at final time
    plot_vector(u1, title=f"Velocity |u| — Level {lvl_idx + 1} at t={T:.1f}")

    results.append({
        "lvl": lvl_idx + 1, "res": resolution, "segs": segs, "cells": n_cells,
        "dt": dt_val, "steps": step,
        "t_start": t_start, "t_end": t_end, "T_lift": T_lift,
        "C_D_max": C_D_max, "C_L_max": C_L_max, "St": St_val, "dp": dp_val,
        "wall": t_wall, "periodic": True,
    })

# **Results**

In [ ]:
print("Schäfer-Turek 2D-2 — graded mesh (v6), GLS P1/P1, dt = time_step(msh, U_m=1.5)")
print(f"Reference nominal: C_D,max={C_D_max_ref}  C_L,max={C_L_max_ref}  "
      f"St={St_ref}  dp={dp_ref}")
print("ST96 intervals: C_D,max∈[3.22,3.24]  C_L,max∈[0.99,1.01]  "
      "St∈[0.295,0.305]  Δp∈[2.46,2.50]")
print()

def _flag(val, lo, hi):
    return "\u2713" if lo <= val <= hi else "\u2717"

valid = [r for r in results if r is not None and r.get("periodic")]
if not valid:
    print("No periodic results available (FAST mode or T too short).")
else:
    hdr = (f"{'lvl':>3} {'res':>4} {'cells':>7} {'dt':>8} {'steps':>6}  "
           f"{'C_D,max':>8} {'\u2713':>2}  "
           f"{'C_L,max':>8} {'\u2713':>2}  "
           f"{'St':>6} {'\u2713':>2}  "
           f"{'\u0394p':>7} {'\u2713':>2}  "
           f"{'T_lift':>7}  {'wall':>7}")
    print(hdr)
    print("-" * len(hdr))
    for r in valid:
        print(f"{r['lvl']:>3d} {r['res']:>4d} {r['cells']:>7d} "
              f"{r['dt']:>8.5f} {r['steps']:>6d}  "
              f"{r['C_D_max']:>8.4f} {_flag(r['C_D_max'],3.22,3.24):>2}  "
              f"{r['C_L_max']:>8.4f} {_flag(r['C_L_max'],0.99,1.01):>2}  "
              f"{r['St']:>6.4f} {_flag(r['St'],0.295,0.305):>2}  "
              f"{r['dp']:>7.4f} {_flag(r['dp'],2.46,2.50):>2}  "
              f"{r['T_lift']:>7.4f}  {r['wall']:>6.0f}s")

# **Discussion**

The 2D-2 benchmark exercises the unsteady vortex-shedding regime (Re=100). The GLS P1/P1 scheme with fractional-step time integration is used without modification from the 2D-1 steady case; only the time step and analysis window change.

**Time step:** dt = time_step(msh, U_ref=U_m=1.5, C_CFL=0.5) = 0.5·h_min/1.5. h_min = 3·dt: L1 h=0.00714 m (6301 steps), L2 h=0.00350 m (12850 steps), L3 h=0.00156 m (28847 steps). Refinement ratios: r12=h1/h2≈2.04, r23=h2/h3≈2.24 (graded mesh; not exactly 2).

**St (Strouhal number):** L1=0.2918 ✗, L2=0.2986 ✓, L3=0.3011 ✓. Monotone increase; enters interval at L2. Differences: 0.0068 (L1→L2), 0.0025 (L2→L3). Apparent convergence order ≈ 1.5 (three-point Richardson with actual h values).

**C_D,max:** L1=3.290, L2=3.268, L3=3.241. Monotone decrease. Differences: 0.0219 (L1→L2), 0.0276 (L2→L3). UNEXPLAINED: the L2→L3 difference is larger than L1→L2 despite a finer mesh; apparent order ≈ −0.1. Pre-asymptotic behavior is the leading candidate but is not verified. C_D,max at L3 = 3.241 remains just above the upper bound 3.240 (+0.001).

**C_L,max:** L1=0.778 ✗, L2=1.035 ✗, L3=1.001 ✓. Non-monotone: the large L1→L2 jump (+0.257) reflects wake under-resolution at L1; L2 overshoots (+0.025); L3 enters the interval. Standard convergence rate not estimated (non-monotone). UNEXPLAINED: the L2 overshoot is inconsistent with an over-dissipation argument (over-dissipation would reduce amplitude, not increase it).

**Δp:** L1=2.522, L2=2.505, L3=2.488. Monotone decrease; enters interval at L3. Differences: 0.0178 (L1→L2), 0.0162 (L2→L3) — nearly equal. Apparent convergence order ≈ 0.3 (sub-linear).

**T_lift:** 0.3427 → 0.3349 → 0.3321 s. Monotone decrease. Apparent convergence order ≈ 1.6.

**Wall time:** L3 ran via nbconvert over ~32 h wall-clock (114797 s as reported by the kernel; run 2026-10-07–08, affected by sleep/suspend). Per CLAUDE.md convention, future runs projected >1 h should use a standalone script under `caffeinate -i` with CPU-time logging.